# Week 2 Day 06 — LLM API Experiments

## Objectives

- Call an LLM API from Jupyter
- Load the API key using an environment variable
- Sweep temperature from 0 to 1
- Count tokens for several inputs
- Relate token usage to cost
- Trigger and document a hallucination
- Improve the prompt to reduce hallucination

In [1]:
%pip install openai python-dotenv tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully.")

API key loaded successfully.


## 1. Load the API key securely

The real API key is stored in `.env`.

Never hard-code the API key inside the notebook.

## 2. Create the LLM client

Create the OpenAI client using the API key loaded from the environment.

In [3]:
import openai

client = openai.OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

print("Client created successfully.")

Client created successfully.


## 3. First real LLM API call

Send a simple prompt to the model and display its response.

In [4]:
prompt = "Explain what an API is to a beginner in three sentences."
Model = "openai/gpt-oss-20b"
response = client.responses.create(
    model=Model,
    input=prompt
)

print(response.output_text)

An API, or Application Programming Interface, is like a menu in a restaurant: it lists all the functions a software can offer and shows how to request them. Developers use APIs so their programs can ask other software for services or data without needing to know how those services are implemented. By calling an API, a program can perform tasks—such as sending an email, fetching a map, or posting a tweet—by sending a request and receiving a response.


## 2. Temperature Sweep

Temperature controls the randomness/variation of the model's output.

We will use the same prompt with temperatures from 0 to 1 and compare the responses.

In [5]:
prompt = "Write a creative description of a rainy evening in Chennai in 50 words."

temperatures = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

for temperature in temperatures:
    response = client.responses.create(
        model=Model,
        input=prompt,
        temperature=temperature
    )

    print("=" * 60)
    print("Temperature:", temperature)
    print(response.output_text)

Temperature: 0
Rain drizzles over Chennai’s neon‑lit streets, turning the bustling harbor into a silver mirror. The air hums with distant traffic, while street vendors offer steaming idlis, their aromas mingling with petrichor. Lanterns flicker, casting warm halos on wet pavements, and the city breathes, alive in rhythmic drops under soft sky.
Temperature: 0.2
Rain drummed against the sweltering streets of Chennai, turning the neon‑lit lanes into shimmering mirrors. The scent of fried dosas mingled with petrichor, while street vendors shouted over the hiss of monsoon clouds. Lamps flickered, casting amber halos on wet pavements, and the city exhaled alive in silky silence today.
Temperature: 0.4
Rain drummed on the slatted eaves of colonial bungalows, turning the bustling streets of Chennai into shimmering silver ribbons. The air, heavy with jasmine and sea salt, carried the distant hum of trams and the scent of dosas. Lanterns flickered, reflecting in puddles, while lovers whispered p

## 3. Token Counting and Cost

LLMs process text as tokens rather than ordinary words.

More tokens generally means more usage and, for paid APIs, potentially higher cost.

In [6]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

inputs = [
    "Hello world!",
    "Explain Python functions to a beginner.",
    "Large language models generate text one token at a time.",
    "PostgreSQL, Redis, FastAPI, Docker, and Python are backend technologies."
]

for text in inputs:
    token_count = len(encoding.encode(text))

    print("-" * 60)
    print("Text:", text)
    print("Token count:", token_count)

------------------------------------------------------------
Text: Hello world!
Token count: 3
------------------------------------------------------------
Text: Explain Python functions to a beginner.
Token count: 8
------------------------------------------------------------
Text: Large language models generate text one token at a time.
Token count: 11
------------------------------------------------------------
Text: PostgreSQL, Redis, FastAPI, Docker, and Python are backend technologies.
Token count: 16


### Cost relationship

For a paid API:

**Input cost = input tokens × input token price**

**Output cost = output tokens × output token price**

The exact price depends on the model.

## 4. Trigger a Hallucination

We will give the model a false premise and observe whether it invents information.

In [7]:
prompt = """
Who was Dr. Arvind Ramanathan, the famous Indian computer scientist
who invented the Ramanathan Algorithm in 1987?

Give his biography, university, awards, and major publications.
"""

response = client.responses.create(
    model=Model,
    input=prompt,
    reasoning={
        "effort": "medium"
    }
)

print(response.output_text)



I’m sorry, but I can’t find any reliable information about a computer scientist named **Dr. Arvind Ramanathan** who invented a “Ramanathan Algorithm” in 1987. A search of major bibliographic databases (IEEE Xplore, ACM Digital Library, Google Scholar, Web of Science, Scopus) and reputable biographical resources (University directories, National Academy listings, and award records) does not return any publications, patents, or citations that match that name or that algorithm.  

There is no record of Dr. Ramanathan receiving prominent national or international computer‑science awards, nor is there a documented university affiliation that corresponds to the details you provided. It is possible that:

| Scenario | What might be happening |
|----------|------------------------|
| **Name confusion** | The scientist’s name could be similar but not identical (e.g., “Arvind Ramanathan” vs. “Arvind Raman”). |
| **Misattributed algorithm** | The algorithm might be known by a different name, or i

## Hallucination Observation

Review the response above.

Look for:

- Invented biography
- Invented university
- Invented awards
- Invented publications
- Unsupported dates or achievements

A fluent and confident answer does not necessarily mean the information is true.

## 5. Fix the Hallucination With Better Prompting

Now explicitly tell the model not to assume the claim is true or invent facts.

In [8]:
prompt = """
Analyze this claim carefully:

Dr. Arvind Ramanathan was a famous Indian computer scientist
who invented the Ramanathan Algorithm in 1987.

Do not assume this claim is true.
Do not invent facts.

If you cannot verify that this person or algorithm exists,
say so clearly.

Separate known information from assumptions in the question.
"""

response = client.responses.create(
    model=Model,
    input=prompt,
    temperature=0
)

print(response.output_text)

**Known information (verified from reputable sources)**  
- No widely recognized record of a computer scientist named **Dr. Arvind Ramanathan** appears in major academic databases, professional societies, or historical accounts of computer science.  
- There is no documented algorithm called the **“Ramanathan Algorithm”** in the literature of computer science, mathematics, or related fields, nor is there any reference to its invention in 1987.

**Assumptions made in the claim**  
- The claim assumes that Dr. Arvind Ramanathan was a *famous* Indian computer scientist.  
- It further assumes that he *invented* an algorithm named after him in the year 1987.

**Conclusion**  
Because there is no verifiable evidence supporting the existence of either the individual or the algorithm, I cannot confirm the claim. If you need definitive confirmation, you would need to consult primary sources such as academic publications, patent records, or institutional archives that could substantiate the exi

## Hallucination Experiment — Final Observation

The improved prompt produced a safer response.

The model did not accept the fictional premise as fact. Instead, it:

- Identified the claim as unverified.
- Separated known information from assumptions.
- Avoided inventing a biography, university, awards, or publications.
- Clearly stated that the claim could not be confirmed.
- Recommended checking primary sources for verification.

### Conclusion

Better prompting can reduce hallucination by explicitly instructing the model
to question assumptions, avoid inventing facts, and communicate uncertainty.

However, this does not guarantee factual accuracy. Important information
should still be verified using reliable external sources.